# Description

In this notebook, we benchmark PySR algorithm on the Korns benchmarks.

In [1]:
from config.korns_config import BENCH, PYSR, FEATURE_NAMES
from src.korns_core import RunConfig, load_korns_hdf5, run_benchmark, init_results_csv, append_results_csv_row, SRFitResult

from pysr import PySRRegressor
from dataclasses import dataclass
from typing import Optional, Dict, Any
import numpy as np
import sympy as sp

import warnings
warnings.filterwarnings(
    "ignore",
    message=r"You are using the `\^` operator, but have not set up `constraints` for it\.",
    category=UserWarning,
    module=r"pysr\.sr",
)
warnings.filterwarnings(
    "ignore",
    message=r"Note: it looks like you are running in Jupyter\. The progress bar will be turned off\.",
    category=UserWarning,
    module=r"pysr\.sr",
)


@dataclass
class PySRKornsRegressor:
    name: str = "pysr"
    niterations: int = PYSR.niterations
    populations: int = PYSR.populations
    maxsize: int = PYSR.maxsize
    timeout_in_seconds: Optional[int] = PYSR.timeout_in_seconds

    def fit_predict(self, X_train, y_train, X_test) -> SRFitResult:
        model = PySRRegressor(
            niterations=self.niterations,
            populations=self.populations,
            maxsize=self.maxsize,
            unary_operators=list(PYSR.unary_ops),
            binary_operators=list(PYSR.binary_ops),
            elementwise_loss=PYSR.elementwise_loss,
            model_selection=PYSR.model_selection,
            verbosity=PYSR.verbosity,
            progress=PYSR.progress,
            temp_equation_file=PYSR.temp_equation_file,
            delete_tempfiles=PYSR.delete_tempfiles,
            timeout_in_seconds=self.timeout_in_seconds,
        )
        model.fit(X_train, y_train)
    
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)
    
        try:
            expr = model.sympy()
        except Exception:
            expr = None
    
        return SRFitResult(
            expr=expr,
            y_pred_train=np.asarray(y_pred_train, dtype=np.float64).reshape(-1),
            y_pred_test=np.asarray(y_pred_test, dtype=np.float64).reshape(-1),
            metadata=None,
        )


cfg = RunConfig(
    hdf5_path=BENCH.hdf5_path,
    test_size=BENCH.test_size,
    split_seed=BENCH.split_seed,
    per_problem_seed_offset=BENCH.per_problem_seed_offset,
    algo_seed_offset=BENCH.algo_seed_offset,
    run_seed_offset=BENCH.run_seed_offset,
)

datasets = load_korns_hdf5(cfg.hdf5_path)

# rows = run_benchmark(
#     datasets=datasets,
#     algorithms=[PySRKornsRegressor()],
#     config=cfg,
#     n_runs=BENCH.n_runs,
#     feature_names=FEATURE_NAMES,
# )

# save_results_csv(rows, PYSR.results_csv_path)

rows = run_benchmark(
    datasets=datasets,
    algorithms=[PySRKornsRegressor()],
    config=cfg,
    n_runs=BENCH.n_runs,
    feature_names=FEATURE_NAMES,
    results_csv_path=PYSR.results_csv_path,
)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
[PROBLEM] P1
[GT] 24.3*x3 + 1.57
[ALGO] pysr
[RUN START] run_id=0 seed=28732350661000
[PRED] x3*24.3 - 1*(-1.5699998)
[PROBLEM] P2
[GT] 0.23 + 4.73333333333333*(x1 + x3)/x4
[ALGO] pysr
[RUN START] run_id=0 seed=28732350662000
[PRED] -1*(-0.23000002) + (x1 + x3)*4.7333336/x4
[PROBLEM] P3
[GT] -5.41 + 1.63333333333333*(-x0 + x1/x4 + x3)/x4
[ALGO] pysr
[RUN START] run_id=0 seed=28732350663000
[PRED] x1 - (-1*(-5.2828846) + (x1 + x1 + x1 + tanh(x1))/((x4*x4*(-2.6060328))))
[PROBLEM] P4
[GT] 0.13*sin(x2) - 2.3
[ALGO] pysr
[RUN START] run_id=0 seed=28732350664000
[PRED] exp(x2)*(-0.0012557863) - 1*2.278247
[PROBLEM] P5
[GT] 2.13*log(x4) + 3.0
[ALGO] pysr
[RUN START] run_id=0 seed=28732350665000
[PRED] 2.12960966256718*log(x4) + 3.0003033
[PROBLEM] P6
[GT] 0.13*sqrt(x0) + 1.3
[ALGO] pysr
[RUN START] run_id=0 seed=28732350666000
[PRED] 1.26370566671836*(x0 + log(x0 + 1.4346408) + 1